In [40]:
import sys

sys.path.insert(0, "/home/sppradhan/Desktop/Research/pyBKT/source-py")

In [41]:
from pyBKT.models import Model
import pickle
import pandas as pd

In [42]:
data = pickle.load(open("./data_dict.pkl", "rb"))

In [43]:
data

{'nStudents': 30,
 'nProblems': 20,
 'correctness': array([[0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1],
        [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1],
        [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0],
        [0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [44]:
# Initialize the model with an optional seed
model = Model(seed=42, num_fits=1)

# # Fetch Assistments and CognitiveTutor data (optional - if you have your own dataset, that's fine too!)
# model.fetch_dataset(
#     'https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv', '.')
# model.fetch_dataset(
#     'https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/ct.csv', '.')

# Train a simple BKT model on all skills in the CT dataset

In [45]:
correctness = []
stu_id = []
for i in range(len(data["correctness"])):
    correctness.extend(data["correctness"][i])
    stu_id.extend([i] * len(data["correctness"][i]))

In [46]:
df_data = {}
df_data["Correct First Attempt"] = correctness
df_data["KC(Default)"] = ["KC1"] * len(correctness)
df_data["user_id"] = stu_id

In [47]:
# df = pd.read_csv('ct.csv')
# df.head()

In [48]:
df = pd.DataFrame(df_data)

In [49]:
model.fit(data=df, forgets=True, skills=["KC1"])

(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)
(1, 600)


In [50]:
model.params()

value
skill param   class          
KC1   prior   default 0.47061
      learns  default 0.02571
      guesses default 0.16377
      slips   default 0.11560
      forgets default 0.00440

In [51]:
model.params()

value
skill param   class          
KC1   prior   default 0.47061
      learns  default 0.02571
      guesses default 0.16377
      slips   default 0.11560
      forgets default 0.00440

In [52]:
from sklearn.metrics import roc_auc_score, root_mean_squared_error, accuracy_score

In [53]:
root_mean_squared_error(data["states"].flatten(), preds["state_predictions"].values)

0.2579584704384151

In [54]:
roc_auc_score(data["states"].flatten(), preds["state_predictions"].values)

np.float64(0.964408615495707)

In [55]:
data["correctness"][0]

array([0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1])

In [56]:
model.params()

value
skill param   class          
KC1   prior   default 0.47061
      learns  default 0.02571
      guesses default 0.16377
      slips   default 0.11560
      forgets default 0.00440

In [58]:
import numpy as np
from numba import njit, prange


@njit(fastmath=True, parallel=True)
def get_p_hidden_state_batch_numba(
    prior,  # (n_students,)
    p_T,  # (n_students,)
    p_F,  # (n_students,)
    p_G,  # (n_students,)
    p_S,  # (n_students,)
    correctness,  # (n_students, n_problems)
):
    n_students, n_problems = correctness.shape
    p_hiddens = np.empty((n_students, n_problems + 1), dtype=np.float64)

    for s in prange(n_students):

        # student-specific parameters
        prior_s = prior[s]
        pT = p_T[s]
        pF = p_F[s]
        pG = p_G[s]
        pS = p_S[s]

        one_minus_pS = 1.0 - pS
        one_minus_pG = 1.0 - pG
        one_minus_pF = 1.0 - pF

        p_hiddens[s, 0] = prior_s

        for t in range(n_problems - 1):
            p_L = p_hiddens[s, t]

            if correctness[s, t]:
                num = p_L * one_minus_pS
                denom = num + (1.0 - p_L) * pG
            else:
                num = p_L * pS
                denom = num + (1.0 - p_L) * one_minus_pG

            p_L_given_obs = num / denom

            # transition update
            p_hiddens[s, t + 1] = (
                p_L_given_obs * one_minus_pF + (1.0 - p_L_given_obs) * pT
            )

    return p_hiddens

In [ ]:
@njit(fastmath=True, parallel=True)
def get_p_hidden_state_batch_numba_smoothed(
    prior,    # (n_students,)
    p_T,      # (n_students,)
    p_F,      # (n_students,)
    p_G,      # (n_students,)
    p_S,      # (n_students,)
    correctness,  # (n_students, n_problems)
):
    n_students, T = correctness.shape
    p_smooth = np.empty((n_students, T), dtype=np.float64)

    for s in prange(n_students):
        prior_s = prior[s]
        pT = p_T[s]
        pF = p_F[s]
        pG = p_G[s]
        pS = p_S[s]

        one_minus_pS = 1.0 - pS
        one_minus_pG = 1.0 - pG

        # emission probabilities
        e0 = np.empty(T, dtype=np.float64)  # P(o_t | L_t=0)
        e1 = np.empty(T, dtype=np.float64)  # P(o_t | L_t=1)

        for t in range(T):
            if correctness[s, t] != 0:
                e1[t] = one_minus_pS
                e0[t] = pG
            else:
                e1[t] = pS
                e0[t] = one_minus_pG

        # forward pass (scaled)
        alpha0 = np.empty(T, dtype=np.float64)
        alpha1 = np.empty(T, dtype=np.float64)
        scale = np.empty(T, dtype=np.float64)

        # t = 0
        a0 = (1.0 - prior_s) * e0[0]
        a1 = prior_s * e1[0]
        c0 = a0 + a1
        if c0 == 0.0:
            c0 = 1e-15
        a0 /= c0
        a1 /= c0
        alpha0[0] = a0
        alpha1[0] = a1
        scale[0] = c0

        # transitions:
        # P(L_{t+1}=0 | L_t=0) = 1-pT => (Didn't learn)
        # P(L_{t+1}=0 | L_t=1) = pF   => (Forgot)
        # P(L_{t+1}=1 | L_t=0) = pT   => (Learned)
        # P(L_{t+1}=1 | L_t=1) = 1-pF => (Didn't forget)

        for t in range(1, T):
            prev0 = alpha0[t-1]
            prev1 = alpha1[t-1]

            # predict step
            pL0 = (1.0 - pT) * prev0 + pF * prev1
            pL1 = pT * prev0 + (1.0 - pF) * prev1

            # update with emission
            a0 = pL0 * e0[t]
            a1 = pL1 * e1[t]
            ct = a0 + a1
            if ct == 0.0:
                ct = 1e-15
            a0 /= ct
            a1 /= ct

            alpha0[t] = a0
            alpha1[t] = a1
            scale[t] = ct

        # backward pass (scaled)
        beta0 = np.empty(T, dtype=np.float64)
        beta1 = np.empty(T, dtype=np.float64)

        beta0[T-1] = 1.0
        beta1[T-1] = 1.0

        for t in range(T - 2, -1, -1):
            # from t to t+1
            b0_next = beta0[t+1]
            b1_next = beta1[t+1]

            # beta_t(j) = sum_i A[i,j] * e_i[t+1] * beta_{t+1}(i) * (1/scale[t+1])
            # j = 0
            b0 = (1.0 - pT) * e0[t+1] * b0_next + pT * e1[t+1] * b1_next
            # j = 1
            b1 = pF * e0[t+1] * b0_next + (1.0 - pF) * e1[t+1] * b1_next

            ct_inv = 1.0 / scale[t+1]
            b0 *= ct_inv
            b1 *= ct_inv

            beta0[t] = b0
            beta1[t] = b1

        # smoothed probabilities gamma_t(1) prop. to alpha_t(1) * beta_t(1)
        for t in range(T):
            g0 = alpha0[t] * beta0[t]
            g1 = alpha1[t] * beta1[t]
            norm = g0 + g1
            if norm == 0.0:
                norm = 1e-15
            p_smooth[s, t] = g1 / norm

    return p_smooth

In [59]:
data["correctness"].shape

(30, 20)

In [97]:
scale = 1
scale_problems = 1

In [108]:
prior = [0.468249] * data["correctness"].shape[0] * scale
p_T = [0.026649] * data["correctness"].shape[0] * scale
p_F = [0.00507769] * data["correctness"].shape[0] * scale
p_G = [0.16498] * data["correctness"].shape[0] * scale
p_S = [0.117037] * data["correctness"].shape[0] * scale



In [109]:
correctness_array = np.repeat(data["correctness"], scale, axis=0)
correctness_array = np.repeat(correctness_array, scale_problems, axis=1)
print(correctness_array.shape)

(30, 20)


In [110]:
hidden_states = get_p_hidden_state_batch_numba(prior, p_T, p_F, p_G, p_S, correctness_array)

In [111]:
hidden_states_smoothed = get_p_hidden_state_batch_numba_smoothed(prior, p_T, p_F, p_G, p_S, correctness_array)

In [112]:
hidden_states_smoothed[15]

array([7.51348054e-04, 1.31421881e-04, 4.32231378e-05, 3.47756792e-05,
       6.27903412e-05, 2.70828796e-04, 1.72498809e-03, 1.18771353e-02,
       8.27521435e-02, 5.77550349e-01, 6.67659256e-01, 6.82374239e-01,
       7.97973782e-01, 8.18175337e-01, 9.66575242e-01, 9.93695823e-01,
       9.98646947e-01, 9.99522078e-01, 9.99519307e-01, 9.98628508e-01])

In [113]:
hidden_states.shape

(30, 21)

In [114]:
df

,Correct First Attempt,KC(Default),user_id,correct_predictions,state_predictions
0,0,KC1,0,0.50290,0.47061
1,0,KC1,0,0.25879,0.13186
2,1,KC1,0,0.19667,0.04565
3,0,KC1,0,0.32579,0.22483
4,0,KC1,0,0.20924,0.06310
...,...,...,...,...,...
595,1,KC1,29,0.18525,0.02981
596,0,KC1,29,0.28177,0.16375
597,0,KC1,29,0.20071,0.05127
598,0,KC1,29,0.18748,0.03290


In [115]:
scaled_data_df = pd.DataFrame(
    {
        "Correct First Attempt": correctness_array.flatten(),
        "KC(Default)": ["KC1"] * correctness_array.flatten().shape[0],
        "user_id": np.array([[i] * data["correctness"].shape[1] for i in range(scale*data["correctness"].shape[0])]).flatten(),
    }
)

In [116]:
model.predict(data=scaled_data_df)

(1, 600)


{'data': array([[1, 1, 2, 1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2,
        2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2,
        1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1,
        1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1,
        1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1,
        1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 2, 1, 1, 1, 

,Correct First Attempt,KC(Default),user_id,correct_predictions,state_predictions
0,0,KC1,0,0.50290,0.47061
1,0,KC1,0,0.25879,0.13186
2,1,KC1,0,0.19667,0.04565
3,0,KC1,0,0.32579,0.22483
4,0,KC1,0,0.20924,0.06310
...,...,...,...,...,...
595,1,KC1,29,0.18525,0.02981
596,0,KC1,29,0.28177,0.16375
597,0,KC1,29,0.20071,0.05127
598,0,KC1,29,0.18748,0.03290


In [117]:
pybkt_state = model.predict(data=df)["state_predictions"]

(1, 600)
{'data': array([[1, 1, 2, 1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2,
        2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2,
        1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1,
        1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1,
        1, 2, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, 2, 2, 2, 1, 1, 1,
        1, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 2, 